# Baseline: Ordinal Logistic Regression

Train an ordinal logistic regression model (ordered logit) for the ordered outcome: `lower < same < higher`, and output class probabilities for each class.

- **Ordinal logit** can output class probabilities via the model’s predicted category probabilities.
- It assumes the **proportional odds assumption** (the relationship between each pair of outcome groups is the same).
- We emphasize **time-ordered evaluation only** (no random split) to prevent look-ahead bias.


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load data
_PARQUET_URL = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/df_model.parquet'
_CSV_URL     = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/df_model.csv'

try:
    df = pd.read_parquet(_PARQUET_URL)
except Exception:
    df = pd.read_csv(_CSV_URL)

# Sort by meeting date strictly
df['meeting_date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values('meeting_date').reset_index(drop=True)

display(df.head())
print("Class distribution:\n", df['decision'].value_counts())

In [ ]:
# Map to ordered integers (lower -> 0, same -> 1, higher -> 2)
order_map = {'lower': 0, 'same': 1, 'higher': 2}
df['y_ord'] = df['decision'].map(order_map)

y_ord = df['y_ord']

# Define X (numeric features only)
exclude_cols = ['meeting_date', 'decision', 'decision_num', 'y_ord']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].select_dtypes(include=[np.number])

# Fix missing data if any (impute with forward fill then 0)
X = X.fillna(method='ffill').fillna(0)


In [ ]:
# Walk-forward evaluation
MIN_TRAIN = 30

preds_prob = []
preds_class = []
actual_labels = []
date_labels = []

classes = ['lower', 'same', 'higher']

for t in range(MIN_TRAIN, len(df)):
    X_train = X.iloc[:t]
    y_train = y_ord.iloc[:t]
    
    X_test = X.iloc[[t]]
    y_test = y_ord.iloc[t]
    
    try:
        # OrderedModel handles thresholding internally, DO NOT add a constant.
        mod = OrderedModel(y_train, X_train, distr='logit')
        res = mod.fit(method='bfgs', disp=False)
        
        # Predict probability for the single test row
        # Returns array of [P(0), P(1), P(2)]
        pred_proba = res.predict(X_test)[0] 
    except Exception as e:
        # If the model fails to converge (e.g., highly collinear features in a small sample),
        # safely default to placing 1.0 on the majority class of the training set.
        majority = int(y_train.mode()[0])
        pred_proba = np.zeros(3)
        pred_proba[majority] = 1.0
    
    # Defensive check to ensure we got all 3 class probabilities
    if len(pred_proba) < 3:
        temp = np.zeros(3)
        temp[:len(pred_proba)] = pred_proba
        pred_proba = temp
        
    pred_idx = np.argmax(pred_proba)
    
    preds_prob.append(pred_proba)
    preds_class.append(classes[pred_idx])
    actual_labels.append(classes[int(y_test)])
    date_labels.append(df.iloc[t]['meeting_date'])


In [ ]:
preds_prob = np.array(preds_prob)

# Report metrics
acc = accuracy_score(actual_labels, preds_class)
ll = log_loss(actual_labels, preds_prob, labels=classes)
cm = confusion_matrix(actual_labels, preds_class, labels=classes)

print(f"Accuracy: {acc:.4f}")
print(f"Log Loss: {ll:.4f}\n")

print("Confusion Matrix:")
df_cm = pd.DataFrame(cm, index=[f"True_{c}" for c in classes], columns=[f"Pred_{c}" for c in classes])
display(df_cm)


In [ ]:
# Result tables and saving
results_df = pd.DataFrame({
    'meeting_date': date_labels,
    'true_label': actual_labels,
    'pred_label': preds_class,
    'P(lower)': preds_prob[:, 0],
    'P(same)': preds_prob[:, 1],
    'P(higher)': preds_prob[:, 2]
})

display(results_df.head())

# Save to disk (optional — gracefully skipped if directory does not exist)
import os
try:
    output_dir  = '../data'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'preds_ordinal.csv')
    results_df.to_csv(output_path, index=False)
    print(f"Saved prediction table to {output_path}")
except Exception as e:
    print(f"Note: could not save to disk ({e}). Results displayed above.")